# Preprocessing e Modelagem — Predição de Alfabetização (execução real)

**Objetivo deste notebook:** orquestrar, de ponta a ponta e contra dados reais
do BigQuery, o pipeline de modelagem definido nas Tasks 1-10: materializar a
tabela Gold enriquecida (aluno + território), dividir os dados em
treino/validação/teste com separação temporal (2023 desenvolvimento, 2024
out-of-time), treinar e selecionar o melhor modelo, avaliar no teste (duas
janelas temporais), diagnosticar variação temporal, interpretar via
SHAP/feature importance, e agregar o risco previsto por município.

Este notebook consome os módulos de `src/` já testados por `pytest` nas
Tasks 1-10 — aqui eles são executados uma única vez, ponta a ponta, contra o
dado de produção (~3,9 milhões de linhas), não contra fixtures. Por isso não
é coberto por `pytest`: sua validação é a própria execução sem erro,
descrita na Task 12 do plano
(`docs/superpowers/specs/2026-09-11-pipeline-modelagem-design.md`).

As células markdown seguem o mesmo formato **Motivo → Resultado → Decisão**
usado no notebook de EDA (`01_eda_gold_e_alunos.ipynb`), sempre que uma etapa
produzir um resultado que muda ou confirma uma decisão de modelagem.


In [1]:
import os
import sys
from pathlib import Path

# O kernel do Jupyter inicia com cwd = diretório do notebook (notebooks/),
# mas os módulos em `src/` e os caminhos usados neste notebook (ex.:
# "reports/...") são relativos à raiz do projeto — subimos um nível e
# adicionamos a raiz ao sys.path antes de qualquer `from src...`.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
from google.cloud import bigquery

pd.set_option("display.max_columns", 50)

# `src.visualization.graficos` força o backend "Agg" (matplotlib.use) para
# não exigir display gráfico em testes/CI — como isso reconfigura o backend
# do matplotlib para o processo inteiro, reativamos o backend inline DEPOIS
# de importar os módulos do projeto, para os gráficos aparecerem no notebook.
%matplotlib inline

PROJECT_ID = "fiapfase2"
client = bigquery.Client(project=PROJECT_ID)
print("Projeto BigQuery:", client.project)


Projeto BigQuery: fiapfase2


## 1. Materialização da tabela Gold enriquecida (aluno + território)

In [2]:
from src.preprocessing.gold_materialization import materializar_tabela_enriquecida

# Materialização real — grava (WRITE_TRUNCATE) a tabela
# `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido`. Não é uma
# célula para repetir a cada execução do notebook: é uma materialização de
# camada Gold versionada por código (o código-fonte é o que muda; a tabela é
# regravada quando o código muda), documentada em
# ensinamentos/02-modelagem/01-materializacao-de-camada-gold-versionada.md.
materializar_tabela_enriquecida(client)
print("Tabela materializada:", "fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido")


/Users/pedrosaraiva/FIAP-Fase3/venv/lib/python3.12/site-packages/google/cloud/bigquery/_pandas_helpers.py:486: FutureWarning: Loading pandas DataFrame into BigQuery will require pandas-gbq package version 0.26.1 or greater in the future. Tried to import pandas-gbq and got: No module named 'pandas_gbq'
  warnings.warn(


Tabela materializada: fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** o modelo trabalha na granularidade do aluno (decisão da EDA,
>   notebook 01), mas os microdados de aluno (`basedosdados.br_inep_...alunos`)
>   não têm nenhuma variável territorial/de meta — só a camada Gold da Fase 2
>   (`indicador_por_municipio`) tem `taxa_alfabetizacao`, `gap_meta_resultado` e
>   `meta_alfabetizacao_2024` por município/ano. Sem uma tabela que junte as
>   duas, cada notebook teria que refazer esse enriquecimento (e o join tem uma
>   regra sutil de evitar vazamento — ver abaixo) na mão.
> - **Resultado:** `materializar_tabela_enriquecida` roda a extração+join uma
>   única vez e grava o resultado em
>   `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido` — uma tabela
>   "Gold" nova, na granularidade de aluno, contendo `alfabetizado` (target
>   bruto) e as variáveis territoriais do **ano anterior** ao ano do aluno
>   (`taxa_alfabetizacao_ano_anterior`, `gap_meta_resultado_ano_anterior`,
>   `meta_alfabetizacao_ano_anterior`) — usar o indicador do ano anterior (e
>   não do próprio ano) evita vazamento de dados, porque o indicador do ano
>   corrente é calculado a partir do resultado dos próprios alunos daquele
>   ano (ver `ensinamentos/01-eda/04-vazamento-de-dados-e-dados-ausentes.md`
>   e `ensinamentos/02-modelagem/01-materializacao-de-camada-gold-versionada.md`).
> - **Decisão:** todo o restante do notebook lê apenas essa tabela
>   materializada — nenhum outro notebook/módulo repete esse join.


## 2. Carregar dado, preparar o target e dividir em treino/validação/teste

In [3]:
from src.preprocessing.features import preparar_target
from src.preprocessing.splitting import dividir_treino_validacao_teste

# ORDER BY é necessário para reprodutibilidade: `SELECT *` sem ordenação
# explícita retorna as linhas em ordem arbitrária do BigQuery a cada
# execução. Como `train_test_split(..., random_state=42)` só é
# reprodutível se a ORDEM das linhas de entrada também for estável, sem
# este ORDER BY cada execução treinaria em um split ligeiramente
# diferente apesar da seed fixa — foi exatamente isso que causou números
# divergentes entre execuções anteriores deste notebook.
df = client.query(
    "SELECT * FROM `fiapfase2.gold_alfabetizacao.aluno_alfabetizado_enriquecido` "
    "ORDER BY id_aluno, ano"
).to_dataframe()
df["em_risco"] = preparar_target(df)

partes = dividir_treino_validacao_teste(df)
X_treino, y_treino = partes["treino"], partes["treino"]["em_risco"]
X_validacao, y_validacao = partes["validacao"], partes["validacao"]["em_risco"]
X_teste_mesmo_ano, y_teste_mesmo_ano = partes["teste_mesmo_ano"], partes["teste_mesmo_ano"]["em_risco"]
X_teste_futuro, y_teste_futuro = partes["teste_futuro"], partes["teste_futuro"]["em_risco"]

print("df.shape:", df.shape)
print("em_risco (geral):", df["em_risco"].mean())
print("em_risco por ano:")
print(df.groupby("ano")["em_risco"].mean())
print()
for nome, parte in partes.items():
    print(f"{nome}: {parte.shape[0]} linhas, em_risco={parte['em_risco'].mean():.4f}")


df.shape: (3355846, 11)
em_risco (geral): 0.4086301933998163
em_risco por ano:
ano
2023    0.416239
2024    0.402458
Name: em_risco, dtype: Float64

treino: 1052140 linhas, em_risco=0.4162
validacao: 225459 linhas, em_risco=0.4162
teste_mesmo_ano: 225459 linhas, em_risco=0.4162
teste_futuro: 1852788 linhas, em_risco=0.4025


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** a EDA (notebook 01) estimou a proporção de alunos não
>   alfabetizados em ~41% (e alfabetizados ~59%) a partir dos microdados
>   brutos filtrados por presença — antes de qualquer join com a camada
>   territorial. Como `em_risco = 1 - alfabetizado`, essa é a checagem de
>   sanidade natural para confirmar que a materialização (Etapa 1) e o join
>   não introduziram viés de seleção (ex.: perda de linhas em municípios sem
>   indicador do ano anterior).
> - **Resultado:** `df.shape = (3.355.846, 11)`. `em_risco` geral = 40,86%,
>   muito próximo do ~41% estimado na EDA. Por ano: 2023 = 41,62%,
>   2024 = 40,25% — estável entre os dois anos, sem indício de viés de
>   seleção introduzido pela materialização.
> - **Decisão:** a proporção confirmou o esperado e é estável entre
>   2023/2024, então seguimos com o split temporal como definido (70/15/15
>   dentro de 2023 para treino/validação/teste-mesmo-ano, 2024 inteiro como
>   teste out-of-time). Não houve necessidade de investigar problema no
>   join.


## 3. Treinar candidatos e selecionar o vencedor (critério: PR-AUC na validação)

In [4]:
from src.preprocessing.features import build_preprocessor
from src.modeling.candidates import construir_candidatos
from src.modeling.selection import selecionar_melhor_modelo

candidatos = construir_candidatos(build_preprocessor())
nome_vencedor, modelo_vencedor, tabela_selecao = selecionar_melhor_modelo(
    candidatos, X_treino, y_treino, X_validacao, y_validacao,
)
display(tabela_selecao)
print("Modelo vencedor:", nome_vencedor)


,modelo,pr_auc_validacao
0,hist_gradient_boosting,0.597992
1,regressao_logistica,0.595835
2,random_forest,0.592547


Modelo vencedor: hist_gradient_boosting


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** PR-AUC (`average_precision_score`) é o critério de seleção
>   porque a classe de interesse (`em_risco=1`) é minoritária (~41%) e é a
>   classe que importa para a decisão de negócio (priorizar municípios/alunos
>   em risco) — ROC-AUC é otimista demais em cenários com esse desbalanceamento
>   moderado, porque pondera bem também o desempenho na classe majoritária.
> - **Resultado:** `hist_gradient_boosting` venceu com PR-AUC de validação
>   0,5980, seguido de perto por `regressao_logistica` (0,5958) e
>   `random_forest` (0,5925) — corrida acirrada, sem um candidato claramente
>   dominante.
> - **Decisão:** seguimos com `hist_gradient_boosting` como modelo final.
>   **Esta é a única célula do notebook em que números de validação
>   aparecem no relatório final** — a partir daqui, toda métrica reportada
>   vem exclusivamente dos conjuntos de teste (mesmo-ano e out-of-time),
>   nunca mais da validação, para não recontaminar a escolha do modelo com
>   informação de teste.


## 4. Avaliação final no teste — 2023 (mesmo ano) e 2024 (out-of-time) — uma única vez

In [5]:
from src.evaluation.metricas import calcular_metricas_teste, tabela_limiares

metricas_teste_mesmo_ano = calcular_metricas_teste(modelo_vencedor, X_teste_mesmo_ano, y_teste_mesmo_ano)
metricas_teste_futuro = calcular_metricas_teste(modelo_vencedor, X_teste_futuro, y_teste_futuro)

print("Teste-2023 (mesmo ano):", metricas_teste_mesmo_ano)
print("2024 (out-of-time):", metricas_teste_futuro)

# Os limiares são escolhidos a partir da VALIDAÇÃO, nunca do teste-2024 —
# ver disciplina "teste avaliado uma única vez, nunca usado para escolher
# nada". Aqui `tabela_limiares` serve só para obter os 3 limiares nomeados
# (padrão/F1-ótimo/recall-prioritário); a precisão/recall mostrada nesta
# tabela é a da PRÓPRIA validação (usada só para localizar o limiar) — o
# número que realmente reportamos como resultado é a precisão/recall
# desses mesmos limiares aplicados ao teste-2024, calculado na célula
# seguinte.
limiares_validacao = tabela_limiares(modelo_vencedor, X_validacao, y_validacao, recall_alvo=0.8)
display(limiares_validacao)


Teste-2023 (mesmo ano): {'roc_auc': 0.6877898400805447, 'pr_auc': 0.5962358911279991}
2024 (out-of-time): {'roc_auc': 0.6388377676351567, 'pr_auc': 0.5277143923398964}


,cenario,limiar,precisao,recall
0,padrao (limiar 0.5),0.500000,0.542066,0.66466
1,otimizado para F1,0.391674,0.484413,0.88270
2,recall-prioritario (recall >= 80%),0.433521,0.507543,0.80017


**Por que escolher o limiar na validação e não no teste-2024:** os três
cenários de limiar acima (padrão, F1-ótimo, recall-prioritário) foram
localizados escaneando `precision_recall_curve` sobre a **validação**, não
sobre o teste-2024 — se os limiares fossem escolhidos olhando para as
próprias respostas de 2024, o teste deixaria de ser uma avaliação
independente (o modelo "aprenderia", por meio da escolha humana do
limiar, informação do conjunto que deveria medir sua generalização). A
tabela acima mostra os limiares e a precisão/recall *na validação*
(usada apenas para localizá-los); a tabela seguinte aplica esses mesmos
três limiares — já fixados — ao teste-2024 e reporta a precisão/recall
resultante, que é o número final citado no README.


In [6]:
from src.evaluation.metricas import _precisao_recall_em_limiar

# `_precisao_recall_em_limiar` é uma função privada de `src/evaluation/metricas.py`
# (prefixo `_`) — reaproveitada diretamente aqui porque este notebook é
# código de análise, não uma API pública consumida por outro módulo; não
# há razão para duplicar a lógica ou promovê-la a função pública só para
# esta célula.
probabilidades_teste_futuro = modelo_vencedor.predict_proba(X_teste_futuro)[:, 1]

linhas_2024 = []
for _, linha in limiares_validacao.iterrows():
    precisao_2024, recall_2024 = _precisao_recall_em_limiar(
        y_teste_futuro, probabilidades_teste_futuro, linha["limiar"]
    )
    linhas_2024.append({
        "cenario": linha["cenario"],
        "limiar (escolhido na validacao)": linha["limiar"],
        "precisao_2024": precisao_2024,
        "recall_2024": recall_2024,
    })

tabela_limiares_2024 = pd.DataFrame(linhas_2024)
display(tabela_limiares_2024)


,cenario,limiar (escolhido na validacao),precisao_2024,recall_2024
0,padrao (limiar 0.5),0.500000,0.482562,0.665154
1,otimizado para F1,0.391674,0.445240,0.868747
2,recall-prioritario (recall >= 80%),0.433521,0.460549,0.795873


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** o teste-mesmo-ano (2023, held-out do mesmo ano de treino) mede
>   a capacidade de generalização "normal" do modelo; o teste-2024
>   (out-of-time, um ano inteiro nunca visto em treino/validação) mede a
>   capacidade de generalizar para o futuro — o cenário real de uso do
>   modelo (prever risco no ano seguinte ao treinado). Os limiares de
>   decisão, por sua vez, são escolhidos **apenas na validação** — nunca no
>   teste-2024 — pela mesma disciplina de não deixar nenhum conjunto de
>   teste influenciar uma escolha feita antes de reportar o resultado final.
> - **Resultado:** teste-2023: ROC-AUC 0,6878 / PR-AUC 0,5962. 2024
>   (out-of-time): ROC-AUC 0,6388 / PR-AUC 0,5277. Os três limiares nomeados
>   foram localizados escaneando a validação: padrão (0,5), F1-otimizado
>   (limiar 0,3917) e recall-prioritário ≥80% (limiar 0,4335). Aplicando
>   esses mesmos limiares — já fixados — ao teste-2024: padrão → precisão
>   0,483/recall 0,665; F1-otimizado → precisão 0,445/recall 0,869;
>   recall-prioritário → precisão 0,461/recall 0,796.
> - **Decisão:** há uma queda real de desempenho de 2023 para 2024 (~0,05
>   em ROC-AUC, ~0,07 em PR-AUC) — investigada na Etapa 5 abaixo, que
>   confirma covariate shift (mudança de composição da amostra) como causa
>   principal (com uma ressalva sobre o vazamento fraco do fallback de ano,
>   ver "Limitações do projeto" no README). Os três cenários de limiar (não
>   um único "oficial") alimentam a seção "Aplicação prática para políticas
>   públicas" do README — sempre reportando o par (limiar escolhido na
>   validação, resultado medido no teste-2024), nunca um limiar escolhido
>   olhando para o próprio 2024.


## 4b. Comparação real vs. previsto em 2024 — matriz de confusão e calibração


In [7]:
from src.evaluation.metricas import matriz_confusao_em_limiar, tabela_calibracao
from src.visualization.graficos import plot_calibracao

print("Matrizes de confusão em 2024, para os limiares escolhidos na validação:\n")
for _, linha in limiares_validacao.iterrows():
    matriz = matriz_confusao_em_limiar(y_teste_futuro, probabilidades_teste_futuro, linha["limiar"])
    print(f"{linha['cenario']} (limiar {linha['limiar']:.4f}):")
    print(f"  Verdadeiro positivo: {matriz['verdadeiro_positivo']:>8}  |  Falso positivo:      {matriz['falso_positivo']:>8}")
    print(f"  Falso negativo:      {matriz['falso_negativo']:>8}  |  Verdadeiro negativo: {matriz['verdadeiro_negativo']:>8}")
    print()

calibracao_2024 = tabela_calibracao(y_teste_futuro, probabilidades_teste_futuro, n_faixas=10)
display(calibracao_2024)
fig_calibracao = plot_calibracao(calibracao_2024)
fig_calibracao.savefig("reports/calibracao_2024.png", dpi=100, bbox_inches="tight")


Matrizes de confusão em 2024, para os limiares escolhidos na validação:

padrao (limiar 0.5) (limiar 0.5000):
  Verdadeiro positivo:   495985  |  Falso positivo:        531832
  Falso negativo:        249684  |  Verdadeiro negativo:   575287

otimizado para F1 (limiar 0.3917):
  Verdadeiro positivo:   647798  |  Falso positivo:        807143
  Falso negativo:         97871  |  Verdadeiro negativo:   299976

recall-prioritario (recall >= 80%) (limiar 0.4335):
  Verdadeiro positivo:   593458  |  Falso positivo:        695130
  Falso negativo:        152211  |  Verdadeiro negativo:   411989



,faixa,probabilidade_media_prevista,taxa_real_observada,n_alunos
0,"(0.0015300000000000001, 0.306]",0.187019,0.184398,185626
1,"(0.306, 0.374]",0.347519,0.295683,193224
2,"(0.374, 0.425]",0.407283,0.326579,177259
3,"(0.425, 0.48]",0.457454,0.373107,185834
4,"(0.48, 0.509]",0.501451,0.378177,186714
5,"(0.509, 0.542]",0.526357,0.389332,185546
6,"(0.542, 0.583]",0.559826,0.449874,183027
7,"(0.583, 0.64]",0.606737,0.480226,185015
8,"(0.64, 0.692]",0.670360,0.538569,206447
9,"(0.692, 0.944]",0.753376,0.620722,164096


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** ROC-AUC/PR-AUC e a tabela de limiares resumem a qualidade
>   do modelo em números agregados, mas não mostram de forma direta "o
>   modelo previu X, a realidade foi Y" — a comparação mais concreta pra
>   demonstrar (ou questionar) a capacidade real do modelo. A matriz de
>   confusão mostra as 4 contagens brutas por trás de precisão/recall; a
>   curva de calibração responde uma pergunta diferente e complementar:
>   quando o modelo diz "70% de risco", isso corresponde a uma frequência
>   real de ~70% entre os alunos daquela faixa de probabilidade, ou o
>   modelo está sistematicamente otimista/pessimista?
> - **Resultado:** nas 3 matrizes de confusão, o número de falsos
>   positivos supera o de verdadeiros positivos em todos os cenários (ex.:
>   limiar padrão — 531.832 falsos positivos vs. 495.985 verdadeiros
>   positivos) — um primeiro sinal de que o modelo alarma mais do que
>   deveria em 2024. A curva de calibração confirma isso de forma direta:
>   o modelo fica **sistematicamente abaixo da diagonal** em toda a faixa de
>   probabilidade — quando ele diz "50% de risco", a taxa real observada é
>   de só ~38%; quando diz "75%", a taxa real é ~62%. O desvio cresce
>   quanto maior a probabilidade prevista.
> - **Decisão:** o modelo está **descalibrado por superestimação** em
>   2024, não apenas "um pouco menos preciso" — as probabilidades
>   previstas não podem ser lidas literalmente como "chance real de risco"
>   nesse ano, só usadas para *ranquear* (quem está em risco relativo a
>   quem). Essa descalibração é consistente com as duas causas já
>   discutidas na Etapa 5 (covariate shift + vazamento fraco do fallback de
>   2023): o modelo aprendeu, em 2023, um nível de risco "de base" mais
>   alto do que o que 2024 realmente apresenta. Documentado no README como
>   limitação adicional, e como um motivo concreto para não usar o número
>   de risco previsto isoladamente — comparar sempre contra a taxa real
>   observada quando ela existir (ver tabela de risco por município
>   abaixo, que já traz as duas lado a lado).


## 5. Diagnóstico de variação temporal (2023 vs. 2024)

In [8]:
from src.evaluation.variacao_temporal import comparar_distribuicoes_categoricas, comparar_distribuicoes_numericas
from src.preprocessing.features import CATEGORICAL_FEATURES, NUMERIC_FEATURES

df_2023 = df[df["ano"] == 2023]
df_2024 = df[df["ano"] == 2024]

variacao_numerica = comparar_distribuicoes_numericas(df_2023, df_2024, NUMERIC_FEATURES)
variacao_categorica = comparar_distribuicoes_categoricas(df_2023, df_2024, CATEGORICAL_FEATURES)

display(variacao_numerica)
display(variacao_categorica)


,coluna,estatistica_ks,p_valor,distribuicao_mudou
0,taxa_alfabetizacao_ano_anterior,0.067675,0.0,True
1,gap_meta_resultado_ano_anterior,0.072888,0.0,True
2,meta_alfabetizacao_ano_anterior,0.072956,0.0,True


,coluna,categoria,proporcao_2023,proporcao_2024,diferenca_absoluta
0,rede,Estadual,0.087506,0.130168,0.042662
1,rede,Municipal,0.912494,0.869819,0.042675
2,rede,Privada,0.000000,0.000013,0.000013
3,regiao,Centro-Oeste,0.105139,0.084446,0.020693
4,regiao,Nordeste,0.334998,0.256638,0.078361
5,regiao,Norte,0.120280,0.097985,0.022295
6,regiao,Sudeste,0.240994,0.407933,0.166939
7,regiao,Sul,0.198589,0.152999,0.045591
8,sigla_uf,AL,0.021839,0.017393,0.004446
9,sigla_uf,AM,0.031977,0.025684,0.006292


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** um gap entre teste-2023 e teste-2024 pode ter duas causas bem
>   diferentes — *covariate shift* (a distribuição das features de entrada
>   mudou, mas a relação feature→target continua a mesma) ou *concept shift*
>   (a própria relação feature→target mudou) — e a ação corretiva é diferente
>   em cada caso.
>   O teste de Kolmogorov-Smirnov (`estatistica_ks`/`p_valor`) nas features
>   numéricas e a comparação de proporções nas categóricas são a forma de
>   testar a primeira hipótese sem precisar de rótulo (`em_risco`) de 2024.
> - **Resultado:** as 3 features numéricas mudaram de distribuição de forma
>   estatisticamente significativa entre 2023 e 2024 (KS test, p ≈ 0,0 nas
>   três). Nas categóricas, o achado mais revelador: `sigla_uf = SP` tinha
>   ~0% de cobertura em 2023 e passa a 21,6% em 2024 — mecanicamente
>   desloca também `regiao = Sudeste` (24,1% → 40,8%).
> - **Decisão:** classificamos o gap da Etapa 4 como **covariate shift**,
>   não concept shift — a causa é uma mudança na composição geográfica da
>   amostra disponível na fonte pública de dados (São Paulo praticamente
>   não aparecia em 2023 e passou a aparecer em 2024), não uma mudança real
>   no comportamento educacional dos alunos. Essa distinção importa para o
>   README: o modelo não "piorou", está vendo uma população parcialmente
>   diferente.


## 6. Interpretabilidade — importância de features e SHAP

In [9]:
from src.modeling.interpretabilidade import calcular_shap_values
from src.visualization.graficos import plot_importancia_features, plot_curva_precisao_recall

preprocessador_ajustado = modelo_vencedor.named_steps["preprocessamento"]
classificador_vencedor = modelo_vencedor.named_steps["classificador"]
nomes_features = list(preprocessador_ajustado.get_feature_names_out())

# SHAP: amostra (custo computacional), não o dataset completo — ver nota
# sobre amostragem para SHAP em
# ensinamentos/02-modelagem/04-disciplina-validacao-vs-teste-no-loop-de-iteracao.md.
# Calculado ANTES da importância porque, para HistGradientBoostingClassifier,
# o SHAP é a única fonte de importância global disponível (ver célula abaixo).
amostra_shap = X_teste_futuro.sample(n=min(3000, len(X_teste_futuro)), random_state=42)
valores_shap = calcular_shap_values(modelo_vencedor, amostra_shap)


**Checagem de sanidade da correção (commit `f815d56`):** antes da tentativa
anterior, as 3 features numéricas territoriais ficavam 100% nulas no
cohort de treino (2023), porque `REDE_REFERENCIA_TERRITORIO` era
"Pública (Estadual e Municipal)" (sem `gap_meta_resultado`/
`meta_alfabetizacao_2024` na Gold real) e o join usava sempre `ano-1`
(2022, inexistente na Gold). A correção trocou a rede de referência para
"Municipal" e adicionou fallback para o mesmo ano quando `ano-1` não
existir. A célula abaixo verifica isso diretamente nos dados de treino
antes de prosseguir para o SHAP — se ainda houver colunas 100% nulas, o
`SimpleImputer` as descarta silenciosamente do pipeline (comportamento
observado na tentativa anterior).


In [10]:
from src.preprocessing.features import NUMERIC_FEATURES

percentual_nulos_treino = X_treino[NUMERIC_FEATURES].isna().mean()
print("Percentual de nulos nas features numéricas (X_treino, cohort 2023):")
print(percentual_nulos_treino)


Percentual de nulos nas features numéricas (X_treino, cohort 2023):
taxa_alfabetizacao_ano_anterior    0.001127
gap_meta_resultado_ano_anterior    0.040859
meta_alfabetizacao_ano_anterior    0.040859
dtype: float64


In [11]:
import numpy as np
import pandas as pd

# HistGradientBoostingClassifier não expõe `feature_importances_` nem
# `coef_` (só RandomForest e LogisticRegression, respectivamente, expõem —
# lacuna conhecida da API do scikit-learn para esse modelo). Para os três
# candidatos funcionarem com o mesmo código, usamos a importância nativa
# quando disponível e, quando não, a média do |valor SHAP| por feature
# (já calculado na célula anterior) como substituto — mesma unidade
# conceitual (contribuição média para a predição).
if hasattr(classificador_vencedor, "feature_importances_"):
    importancias = classificador_vencedor.feature_importances_
elif hasattr(classificador_vencedor, "coef_"):
    importancias = abs(classificador_vencedor.coef_[0])
else:
    importancias = np.abs(valores_shap.values).mean(axis=0)

tabela_importancia = (
    pd.DataFrame({"feature": nomes_features, "importancia": importancias})
    .sort_values("importancia", ascending=False)
    .reset_index(drop=True)
)
print("Top 10 features mais importantes:")
print(tabela_importancia.head(10).to_string(index=False))

fig_importancia = plot_importancia_features(nomes_features, importancias)
fig_importancia.savefig("reports/importancia_features.png", dpi=100, bbox_inches="tight")

fig_pr = plot_curva_precisao_recall(modelo_vencedor, X_teste_futuro, y_teste_futuro)
fig_pr.savefig("reports/curva_precisao_recall.png", dpi=100, bbox_inches="tight")

import shap
import matplotlib.pyplot as plt

# Nota: `valores_shap` é um shap.Explanation calculado sobre os dados JÁ
# transformados pelo preprocessador (one-hot expande as 3 categóricas em
# várias colunas) — por isso NÃO passamos `amostra_shap` (dataframe bruto,
# menos colunas) como segundo argumento: o shape não bateria com
# `valores_shap.values`. Passar só o Explanation deixa o shap usar seus
# próprios `.data`/`.feature_names` (já no espaço transformado).
shap.summary_plot(valores_shap, show=False)
plt.savefig("reports/shap_summary.png", dpi=100, bbox_inches="tight")
plt.close()

print("Gráficos salvos em reports/importancia_features.png, reports/curva_precisao_recall.png, reports/shap_summary.png")


Top 10 features mais importantes:
                                   feature  importancia
numericas__taxa_alfabetizacao_ano_anterior     0.561655
                categoricas__rede_Estadual     0.037213
numericas__meta_alfabetizacao_ano_anterior     0.029064
numericas__gap_meta_resultado_ano_anterior     0.023580
               categoricas__rede_Municipal     0.012880
               categoricas__regiao_Sudeste     0.012418
              categoricas__regiao_Nordeste     0.010049
                  categoricas__sigla_uf_AM     0.003886
                  categoricas__sigla_uf_CE     0.002319
          categoricas__regiao_Centro-Oeste     0.002234


Gráficos salvos em reports/importancia_features.png, reports/curva_precisao_recall.png, reports/shap_summary.png


> **Motivo → Resultado → Decisão**
>
> - **Motivo:** feature importance (do próprio classificador) dá uma visão
>   rápida e global de quais variáveis o modelo mais usa; SHAP complementa
>   mostrando também a *direção* do efeito de cada feature (valores altos de
>   uma variável aumentam ou diminuem o risco previsto) — informação que a
>   importância bruta não dá, e que é o que o público não técnico
>   (stakeholders do setor público) vai perguntar: "por que este município é
>   classificado como risco alto?".
> - **Resultado:** `taxa_alfabetizacao_ano_anterior` domina amplamente
>   (importância 0,562) — mais da metade do poder preditivo total. Em
>   seguida, com peso bem menor: `rede_Estadual` (0,037),
>   `meta_alfabetizacao_ano_anterior` (0,029), `gap_meta_resultado_ano_anterior`
>   (0,024), `rede_Municipal` (0,013).
> - **Decisão:** a taxa histórica de alfabetização do próprio município é,
>   isoladamente, o preditor mais forte que este modelo encontrou —
>   reforça a leitura de que alfabetização é, em grande parte, um fenômeno
>   territorial/sistêmico, não apenas individual. Essa é a resposta
>   principal que o README (Task 13) destaca para "quais fatores mais
>   afetam a alfabetização".


## 7. Agregação de risco por município e artefatos finais

In [12]:
import joblib
from src.evaluation.risco_por_municipio import comparar_risco_real_previsto_por_municipio

probabilidades_futuro = modelo_vencedor.predict_proba(X_teste_futuro)[:, 1]
risco_por_municipio = comparar_risco_real_previsto_por_municipio(
    X_teste_futuro["id_municipio"], y_teste_futuro, probabilidades_futuro,
)
display(risco_por_municipio.head(20))

risco_por_municipio.to_csv("reports/risco_por_municipio_2024.csv", index=False)
joblib.dump(modelo_vencedor, "reports/modelo_vencedor.joblib")

print("Artefatos salvos em reports/risco_por_municipio_2024.csv e reports/modelo_vencedor.joblib")


,id_municipio,risco_previsto,risco_real,n_alunos,diferenca
0,2919900,0.944146,0.875000,40,0.069146
1,1718501,0.941557,0.500000,48,0.441557
2,2205581,0.935088,0.545455,55,0.389634
3,1717800,0.932110,0.547619,42,0.384491
4,1718006,0.932110,0.617021,47,0.315089
5,1716307,0.932110,0.620690,58,0.311420
6,1715705,0.932110,0.691358,81,0.240752
7,2406908,0.931905,0.571429,21,0.360477
8,3167509,0.928217,0.500000,24,0.428217
9,2907400,0.921042,0.742857,35,0.178185


Artefatos salvos em reports/risco_por_municipio_2024.csv e reports/modelo_vencedor.joblib


## Síntese final (alimenta o README da Task 13)

- **Modelo escolhido:** `hist_gradient_boosting` (PR-AUC de validação
  0,5980, vencendo regressão logística 0,5958 e random forest 0,5925 por
  margem pequena).
- **Métricas de teste:** ROC-AUC/PR-AUC de 0,6878/0,5962 em teste-2023 vs.
  0,6388/0,5277 em 2024 (out-of-time) — gap classificado como **covariate
  shift** (Etapa 5): mudança de composição geográfica da amostra (SP:
  0%→21,6% de cobertura entre 2023 e 2024), com uma ressalva sobre o
  vazamento fraco do fallback de ano no cohort de 2023 (ver "Limitações do
  projeto" no README).
- **Limiares (escolhidos na validação, aplicados uma única vez ao
  teste-2024):** padrão (limiar 0,500) → precisão 0,483/recall 0,665;
  F1-otimizado (limiar 0,392) → precisão 0,445/recall 0,869;
  recall-prioritário ≥80% (limiar 0,434) → precisão 0,461/recall 0,796.
- **Comparação real vs. previsto (Etapa 4b) — o achado mais importante
  desta rodada:** o modelo está **descalibrado por superestimação** em
  2024 — a taxa real observada fica sistematicamente abaixo da
  probabilidade prevista em toda a faixa (ex.: previsto 50% → real ~38%;
  previsto 75% → real ~62%), e as 3 matrizes de confusão mostram mais
  falsos positivos que verdadeiros positivos em todos os cenários de
  limiar. Isso significa que as probabilidades previstas servem bem para
  **ranquear** risco relativo entre municípios, mas não devem ser lidas
  literalmente como "chance real de acontecer" nesse ano.
- **Top-5 municípios de maior risco *previsto* (2024) — com o risco *real*
  observado ao lado, para comparação:** id_municipio 2919900 (previsto
  0,944 / real 0,875 — boa concordância), 1718501 (previsto 0,942 / real
  0,500 — grande superestimação), 2205581 (previsto 0,935 / real 0,545),
  1717800 (previsto 0,932 / real 0,548), 1718006 (previsto 0,932 / real
  0,617). A tabela completa (`reports/risco_por_municipio_2024.csv`) traz
  `risco_previsto`, `risco_real` e a `diferenca` entre os dois para todos
  os municípios — o ranking por risco previsto isoladamente pode enganar
  precisamente pelo viés de superestimação encontrado acima.
- **Top-5 features mais influentes:** `taxa_alfabetizacao_ano_anterior`
  (0,562), `rede_Estadual` (0,037), `meta_alfabetizacao_ano_anterior`
  (0,029), `gap_meta_resultado_ano_anterior` (0,024), `rede_Municipal`
  (0,013).
- **Artefatos gerados:** `reports/risco_por_municipio_2024.csv` (risco
  previsto **e** real por município em 2024, com a diferença entre os
  dois, ordenado pelo risco previsto), `reports/modelo_vencedor.joblib`
  (pipeline completo — pré-processamento + classificador — pronto para
  novas predições), e `reports/importancia_features.png`,
  `reports/curva_precisao_recall.png`, `reports/calibracao_2024.png`,
  `reports/shap_summary.png`.
